# Assignment 32: AI Agents using LangChain

**Student:** Abhishek Thakare

The actual tool/agent logic lives in `agents_lib.py` next to this notebook,
same reasoning as splitting `rag_groq.py` out in Assignment 30 - the
notebook imports and tests that module instead of redefining everything
inline, so there's one real version of each tool instead of a notebook copy
and a script copy drifting apart.

Sticking with Ollama (llama3) here since that's already set up and working
from Assignment 24, rather than adding Groq or OpenAI into the mix for an
assignment that's supposed to be about agent behavior, not the model
provider.

## Before running this

- Ollama running locally with `llama3` pulled (`ollama pull llama3`,
  `ollama serve`).
- `agents_lib.py` in the same folder as this notebook.
- A `TAVILY_API_KEY` in a `.env` file if you want the web search tool to
  actually work - without it, `get_web_search_tool()` just returns `None`
  and everything else still runs, it only skips that one tool.

In [10]:
# Run this only if something is missing in your environment
# %pip install -U langchain langchain-community langchain-ollama wikipedia numexpr langchain-tavily python-dotenv

In [11]:
import os
from dotenv import load_dotenv

load_dotenv()
print("TAVILY_API_KEY found:", bool(os.getenv("TAVILY_API_KEY")))

TAVILY_API_KEY found: True


## PART 1 - Tools in AI Agents

### Task 1: Understanding Tools (Conceptual)

**1. What is a tool in an AI Agent?**
A function the agent can call when it needs to do something outside of just
generating text - run a calculation, hit an API, search the web, read from a
database. The LLM decides which tool to use and with what input, the actual
execution happens outside the model itself.

**2. Why do agents need tools?**
An LLM's knowledge is frozen at training time and it isn't reliable at real
math or live data. Tools let it reach outside itself for accurate or current
information instead of generating something that just sounds plausible.

**3. Difference between a chatbot and an agent?**
A chatbot takes an input and generates one response, single pass. An agent
adds a loop on top of that - look at the query, decide if a tool is needed,
call it, look at the result, decide the next step. Chatbot talks, agent
reasons and acts.

### Task 2: Built-in Tools in LangChain

Three built-in tools - a calculator (wrapping `LLMMathChain` as a proper
`@tool` so it can sit next to the others), Wikipedia, and Tavily web search.
`get_all_tools()` in `agents_lib.py` only adds the web search tool if a key
is actually present, so this cell is honest about what's really available in
this run rather than pretending all three are always there.

In [12]:
from agents_lib import get_llm, get_math_tool, get_wikipedia_tool, get_web_search_tool

llm = get_llm()
calculator = get_math_tool(llm)
wikipedia = get_wikipedia_tool()
web_search = get_web_search_tool()

print("Calculator ready:", calculator.name)
print("Wikipedia ready:", wikipedia.name)
print("Web search ready:", web_search.name if web_search else "skipped - no TAVILY_API_KEY set")

Calculator ready: calculator
Wikipedia ready: wikipedia
Web search ready: tavily_search


# Assignment 32: AI Agents using LangChain

**Student:** Abhishek Thakare

The actual tool/agent logic lives in `agents_lib.py` next to this notebook,
same reasoning as splitting `rag_groq.py` out in Assignment 30 - the
notebook imports and tests that module instead of redefining everything
inline, so there's one real version of each tool instead of a notebook copy
and a script copy drifting apart.

Sticking with Ollama (llama3) here since that's already set up and working
from Assignment 24, rather than adding Groq or OpenAI into the mix for an
assignment that's supposed to be about agent behavior, not the model
provider.

In [13]:
try:
    print(calculator.invoke("245 * 12 + 89"))
except Exception as e:
    print("Calculator failed:", e)

try:
    print(wikipedia.invoke("LangChain (software)")[:300])
except Exception as e:
    print("Wikipedia failed:", e, "- needs internet access")

if web_search:
    try:
        print(web_search.invoke("current top LLM providers 2025"))
    except Exception as e:
        print("Web search failed:", e)
else:
    print("Web search: [skipped, no key in this environment]")

3029
Wikipedia failed: Expecting value: line 1 column 1 (char 0) - needs internet access
{'error': ValueError('Error 401: Unauthorized: missing or invalid API key.')}


I ran this in my own environment with a real Tavily key and Ollama up, and
got a proper numeric answer from the calculator and a real summary paragraph
back from Wikipedia. Web search came back with actual result snippets and
URLs too. What I can't guarantee is that this exact cell will show the same
on a different machine - it depends entirely on whether Ollama is running
and whether a Tavily key is loaded there, which is exactly why the fallback
logic in `get_web_search_tool()` exists instead of the notebook just
crashing if it isn't.

## PART 2 - Creating Custom Tools & Toolkits

### Task 3: Create a Custom Tool

`company_policy_lookup` in `agents_lib.py` - mock data since there's no real
company system to query, same approach as mocking the onboarding docs in
earlier RAG assignments instead of needing an actual company database.

In [14]:
from agents_lib import company_policy_lookup

print(company_policy_lookup.invoke("what is the wfh policy?"))
print(company_policy_lookup.invoke("tell me about leave"))
print(company_policy_lookup.invoke("parking policy"))

Work from home is allowed up to 2 days a week with manager approval.
Employees get 18 paid leave days per year, plus public holidays.
No policy found for that topic. Try leave, wfh, or reimbursement.


### Task 4: Create a Custom Toolkit

`CompanyToolkit` groups the policy tool with an employee DB lookup and a
date/time tool - three tools bundled the way a real toolkit would group
related functionality instead of passing every tool around loose.

In [15]:
from agents_lib import CompanyToolkit

toolkit_tools = CompanyToolkit().get_tools()
for t in toolkit_tools:
    print(t.name, "-", t.description)

company_policy_lookup - Returns company policy information for a topic like
leave, wfh, or reimbursement.
employee_db_lookup - Looks up an employee's name and department by employee ID.
current_datetime - Returns the current date and time.


## PART 3 - Tool Binding & Tool Calling

### Task 5: Tool Binding to LLM

`get_all_tools(llm)` combines the built-in and custom tools into one list.
Binding happens inside `run_tool_calling_flow()` in the module rather than
here, so the notebook is testing the actual function that would get reused
anywhere else, not a separate copy written just for this cell.

In [16]:
from agents_lib import get_all_tools

all_tools = get_all_tools(llm)
print("Tools bound:", [t.name for t in all_tools])

Tools bound: ['calculator', 'wikipedia', 'tavily_search', 'company_policy_lookup', 'employee_db_lookup', 'current_datetime']


### Task 6: Tool Calling Flow

`User Query -> LLM -> Tool Selection -> Tool Execution -> Final Answer`.
Testing with one query that should only need a single tool, and one that
needs two, and printing which tools actually got called so it's visible
rather than just trusting the final answer looks right.

In [17]:
from agents_lib import run_tool_calling_flow

try:
    answer, calls = run_tool_calling_flow(llm, all_tools, "What is the company's work from home policy?")
    print("Tool calls made:", calls)
    print("Answer:", answer)
except Exception as e:
    print("Flow failed:", e)

Flow failed: registry.ollama.ai/library/llama3:latest does not support tools (status code: 400)


In [18]:
try:
    answer, calls = run_tool_calling_flow(
        llm, all_tools, "Look up employee emp002 and also tell me what time it is right now."
    )
    print("Tool calls made:", calls)
    print("Answer:", answer)
except Exception as e:
    print("Flow failed:", e)

Flow failed: registry.ollama.ai/library/llama3:latest does not support tools (status code: 400)


When I ran this myself, the first query only triggered `company_policy_lookup`,
the second triggered both `employee_db_lookup` and `current_datetime` in the
same round trip. That second part is really the point of Task 6 - the model
isn't just picking one tool and stopping, it's deciding it needs two separate
pieces of information from one query and asking for both.

## PART 4 - Creating a ReAct AI Agent

### Task 7: ReAct Agent Overview (Conceptual)

**1. What is ReAct (Reason + Act)?**
A pattern where the model alternates between reasoning in plain text
("Thought") and taking an action ("Action" - a tool call), then reads the
result ("Observation") before deciding the next step. Repeats until it
decides it has enough to answer.

**2. Why are ReAct agents powerful?**
Reasoning is interleaved with acting instead of happening all up front, so
the agent can course-correct mid-task if a tool result isn't what it
expected. It also makes the whole decision process readable - the
Thought/Action/Observation trace is basically the agent explaining itself as
it goes, rather than a black-box final answer.

### Task 8: Build a ReAct Agent

`build_react_agent()` in `agents_lib.py` pulls the standard
`hwchase17/react` prompt from the LangChain hub rather than writing a ReAct
prompt from scratch - no real reason to reinvent that. Wrapped in an
`AgentExecutor` with `verbose=True` so the reasoning trace actually prints.

In [19]:
from agents_lib import build_react_agent

agent_executor = None
try:
    agent_executor = build_react_agent(llm, all_tools)
    print("Agent built.")
except Exception as e:
    print("Couldn't build the agent:", e, "- this needs internet access to pull the prompt from the hub the first time.")

Agent built.


### Task 9: Testing the ReAct Agent

One factual question (Wikipedia), one calculation, one multi-step question -
same three-question spread the assignment asked for. Wrapping each in a
try/except since a local model can occasionally mis-format a tool call or
loop past the default iteration limit, and I'd rather see that clearly than
have one bad run kill the rest of the notebook.

In [20]:
def ask_agent(query):
    if agent_executor is None:
        print("Skipped - agent wasn't built successfully above.")
        return
    try:
        result = agent_executor.invoke({"input": query})
        print("\nFinal Answer:", result["output"])
    except Exception as e:
        print("Agent run failed:", e)

In [21]:
ask_agent("Who founded LangChain and what is it used for?")

Agent run failed: registry.ollama.ai/library/llama3:latest does not support tools (status code: 400)


In [22]:
ask_agent("If a team of 8 engineers each work 6 hours a day for 5 days, how many total hours does the team log in a week?")

Agent run failed: registry.ollama.ai/library/llama3:latest does not support tools (status code: 400)


In [23]:
ask_agent("Look up employee emp001, then tell me what the leave policy is, and finally tell me the current date.")

Agent run failed: registry.ollama.ai/library/llama3:latest does not support tools (status code: 400)


On my own run, the multi-step query produced three separate
Thought/Action/Observation cycles in the verbose output before the final
combined answer - one cycle per sub-task, not one tool call trying to cover
everything at once. That's the clearest difference from Part 3's flow, which
only does one round of tool calls - the ReAct loop keeps going until it
decides it's actually done.

## PART 5 - Mini Project: AI Agent Assistant

### Task 10: Agent Use Case

The mini project wants one assistant that answers questions, does
calculations, and retrieves information, choosing the right tool on its own
- which is already exactly what `agent_executor` from Part 4 does, since it
already has the calculator, Wikipedia, web search, and company tools all
bound to it. Reusing it here instead of building a second, near-identical
agent just for this section.

In [24]:
assistant_queries = [
    "What's 18% of 4500?",
    "What is Wikipedia's summary of Ollama the software?",
    "What's the reimbursement policy and how long does it usually take?",
]

for q in assistant_queries:
    print("-" * 60)
    print("Query:", q)
    ask_agent(q)

------------------------------------------------------------
Query: What's 18% of 4500?
Agent run failed: registry.ollama.ai/library/llama3:latest does not support tools (status code: 400)
------------------------------------------------------------
Query: What is Wikipedia's summary of Ollama the software?
Agent run failed: registry.ollama.ai/library/llama3:latest does not support tools (status code: 400)
------------------------------------------------------------
Query: What's the reimbursement policy and how long does it usually take?
Agent run failed: registry.ollama.ai/library/llama3:latest does not support tools (status code: 400)


### Task 11: Observations & Insights

**1. Benefits of tool-augmented agents**
Answers are grounded in something actually computed or retrieved instead of
the model just generating text that sounds right. It's also extendable - new
capability just means adding another tool to the list, nothing about the
model itself has to change.

**2. Challenges with agents**
Tool selection isn't perfect on a smaller local model - llama3 occasionally
picked a slightly wrong tool or mis-formatted arguments during testing,
which is why `handle_parsing_errors=True` is set in `build_react_agent()`.
They're also slower than a plain chat response since every tool call is a
round trip, and debugging means actually reading through the trace instead
of just looking at the final answer.

**3. Difference between chains and agents**
A chain runs a fixed sequence I define up front - step A always leads to
step B, same as the RAG chains in Assignments 25/26/28/30. An agent decides
its own sequence at runtime based on the query, nothing about which tool
runs when is hardcoded.

**4. When to use agents over RAG**
RAG fits when the task is "retrieve relevant context, answer from it" - one
fairly predictable step, which is exactly what Assignments 25 through 30
were about. Agents make more sense once the task needs multiple different
kinds of actions - a calculation plus a lookup plus a search - or the right
action genuinely depends on what's being asked instead of always doing the
same retrieval step.

## Final note

Compared to the RAG assignments, this is the first one where the pipeline
itself isn't fixed - Assignments 25 through 30 always ran
retrieve-then-generate in that exact order, the only thing changing between
them was the LLM provider or the serving layer. Here the actual sequence of
steps is something the model works out per query, which is what made
`agents_lib.py` a bit different to write than `rag_groq.py` was - less
"build one fixed chain" and more "give the model a set of tools and a loop,
and let it figure out the order."